How many late days are you using for this assignment?

0 late days

In [221]:
# Imports and pip installations (if needed)
!pip install pandas
!pip install numpy
!pip install scikit-learn

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler


[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: C:\Users\<redacted>\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: C:\Users\<redacted>\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: C:\Users\<redacted>\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# Part 1: Load the dataset

In [222]:
# Load the datasets
numerical_df = pd.read_csv('chronic_kidney_disease_categorical.csv')
categorical_df = pd.read_csv('chronic_kidney_disease_numerical.csv')

# Print the data
print(numerical_df)
print(categorical_df)

     unique_id   al   su       rbc        pc         pcc          ba  htn  \
0       224481  3.0  NaN       NaN       NaN  notpresent  notpresent  yes   
1       992643  0.0  0.0       NaN       NaN  notpresent  notpresent   no   
2       308740  1.0  0.0  abnormal  abnormal  notpresent  notpresent  yes   
3       450314  1.0  0.0       NaN    normal  notpresent  notpresent   no   
4       881763  0.0  0.0    normal    normal  notpresent  notpresent   no   
..         ...  ...  ...       ...       ...         ...         ...  ...   
425     769512  0.0  0.0    normal    normal  notpresent  notpresent   no   
426     872034  2.0  1.0  abnormal  abnormal  notpresent     present  yes   
427     643007  NaN  NaN       NaN    normal  notpresent  notpresent  yes   
428     840392  NaN  NaN       NaN       NaN  notpresent  notpresent  yes   
429     856244  0.0  0.0       NaN    normal  notpresent  notpresent  yes   

      dm  cad appet   pe  ane  Target  
0    yes   no  good  yes   no     c

# Part 2: Analyze the Dataset

Refer to this: https://archive.ics.uci.edu/dataset/336/chronic+kidney+disease

Explain what the each data is in your own words. What are the features and labels? Are the features in the given datasets : categorical, numerical or both? Give 3 examples of categorical and numerical columns each (if they exist)

Answer: Features are the inputs, they are the variables attributed to each different piece of data. Labels are what we are tring to predict based on the features.

The features are both categorical and numerical.

Categorical: ba, pc, and rbc

Numerical: age, bp, bgr

# Part 3: Data Preprocessing

A fundamental skill in Machine Learning is mastering the art of data cleaning and preprocessing. In this assignment, you will learn and apply essential data cleaning techniques to transform a raw dataset into a clean, ready-to-use form which you can use for regression or classification tasks. By the end of this assignment, you'll have a fully clean dataset and a solid foundation in preparing data for various machine learning models.

## Part 3.1 : Drop Duplicate rows

Let's start by checking if the given datasets have any duplicate rows (same Unique Id). Use pandas to identify and remove these duplicate rows from the given dataset

In [223]:
# Check for duplicates in the numerical dataset
duplicate_numerical = numerical_df.duplicated().sum()
print(f"Total number of duplicate rows in numerical dataset: {duplicate_numerical}")

# Drop duplicates
numerical_df = numerical_df.drop_duplicates()

# Repeat for categorical dataset
duplicate_categorical = categorical_df.duplicated().sum()
print(f"Total number of duplicate rows in categorical dataset: {duplicate_categorical}")

# Drop duplicates
categorical_df = categorical_df.drop_duplicates()

Total number of duplicate rows in numerical dataset: 30
Total number of duplicate rows in categorical dataset: 30


## Part 3.2: Combine two differents datasets

A good skill to have is to know how to combine 2 different datasets.

Are all the unique ids are present in both datasets? Why do you think so? If not, what do the rows that are missing from one of the datasets look like in the combined table?

Answer: When combined there are 400 rows. When not combined, there were 430 and 420 rows, so there must have been several unique ID's that aren't in both data sets. In the combined data set, the rows that are only in one data set just have the columns that were in the other data set blank.

In [224]:
# Check unique IDs
unique_numerical_ids = set(numerical_df['unique_id'])
unique_categorical_ids = set(categorical_df['unique_id'])

# Check if all IDs match
missing_from_numerical = unique_categorical_ids - unique_numerical_ids
missing_from_categorical = unique_numerical_ids - unique_categorical_ids

print(f"Missing from numerical dataset: {missing_from_numerical}")
print(f"Missing from categorical dataset: {missing_from_categorical}")

# Merge the two datasets
combined_df = pd.merge(numerical_df, categorical_df, on='unique_id', how='outer')
print(combined_df)


Missing from numerical dataset: set()
Missing from categorical dataset: {208034, 240430, 488947, 325238, 254775, 292312, 298297, 663387, 948317, 266335}
     unique_id   al   su     rbc        pc         pcc          ba  htn   dm  \
0       102677  3.0  0.0  normal  abnormal  notpresent  notpresent  yes   no   
1       103570  0.0  0.0  normal    normal  notpresent  notpresent   no   no   
2       104675  2.0  1.0  normal    normal     present  notpresent   no  yes   
3       105923  1.0  0.0     NaN    normal  notpresent  notpresent  yes  yes   
4       107297  0.0  0.0  normal    normal  notpresent  notpresent   no   no   
..         ...  ...  ...     ...       ...         ...         ...  ...  ...   
395     995177  4.0  3.0  normal  abnormal     present     present  yes  yes   
396     996076  0.0  0.0  normal    normal  notpresent  notpresent   no   no   
397     996412  NaN  NaN     NaN       NaN  notpresent  notpresent   no   no   
398     997629  0.0  0.0  normal    normal     

## Part 3.3: Rows with Missing values

Removing missing values from a dataset is important for classification because it ensures the model is trained on complete and accurate data, leading to better performance and reliable predictions. Incomplete data can introduce bias and errors, negatively impacting the model's effectiveness.

In [225]:
# Calculate percentage of rows with missing values
missing_percentage = combined_df.isnull().any(axis=1).mean() * 100
print(f"Percentage of rows with at least one missing value: {missing_percentage}%")

# Drop rows with missing values
cleaned_df = combined_df.dropna()
print(cleaned_df)

Percentage of rows with at least one missing value: 61.75000000000001%
     unique_id   al   su     rbc        pc         pcc          ba  htn   dm  \
0       102677  3.0  0.0  normal  abnormal  notpresent  notpresent  yes   no   
6       109053  0.0  0.0  normal    normal  notpresent  notpresent   no   no   
7       114717  0.0  0.0  normal    normal  notpresent  notpresent   no   no   
8       118191  0.0  0.0  normal    normal  notpresent  notpresent   no   no   
9       118596  0.0  0.0  normal    normal  notpresent  notpresent   no   no   
..         ...  ...  ...     ...       ...         ...         ...  ...  ...   
390     980291  3.0  0.0  normal  abnormal  notpresent  notpresent  yes  yes   
391     991031  0.0  0.0  normal    normal  notpresent  notpresent   no   no   
394     994377  0.0  0.0  normal    normal  notpresent  notpresent   no   no   
395     995177  4.0  3.0  normal  abnormal     present     present  yes  yes   
396     996076  0.0  0.0  normal    normal  notpr

## Part 3.4: Sort the dataset according to the Labels

In [226]:
cleaned_df = cleaned_df.sort_values(by='Target').reset_index(drop=True)
print(cleaned_df)

     unique_id   al   su       rbc        pc         pcc          ba  htn  \
0       102677  3.0  0.0    normal  abnormal  notpresent  notpresent  yes   
1       337834  4.0  0.0    normal  abnormal     present  notpresent  yes   
2       343710  2.0  0.0  abnormal  abnormal  notpresent  notpresent  yes   
3       349892  1.0  3.0  abnormal  abnormal  notpresent  notpresent  yes   
4       353025  3.0  4.0    normal    normal  notpresent  notpresent  yes   
..         ...  ...  ...       ...       ...         ...         ...  ...   
148     486397  0.0  0.0    normal    normal  notpresent  notpresent   no   
149     203466  0.0  0.0    normal    normal  notpresent  notpresent   no   
150     482105  0.0  0.0    normal    normal  notpresent  notpresent   no   
151     228849  0.0  0.0    normal    normal  notpresent  notpresent   no   
152     996076  0.0  0.0    normal    normal  notpresent  notpresent   no   

      dm  cad  ...    bp    bgr     bu    sc    sod  pot  hemo   pcv     wb

## Part 3.5: Encoding Categorical data

In this step, we identify and process the categorical columns in the sorted dataset. We map each unique value in these columns to separate "dummy" columns.

For example, the column 'rbc' will be transformed into two columns 'rbc_normal' and 'rbc_abnormal'. If a row's value in 'rbc' is 'normal', the 'rbc_normal' column will be set to 1 and 'rbc_abnormal' will be set to 0.


**Note: Find a correct pandas function to do this **

In [227]:
# Create dummy variables for categorical columns
categorical_columns = ['rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'Target']
cleaned_df = pd.get_dummies(cleaned_df, columns=categorical_columns, drop_first=False)

print(cleaned_df)


     unique_id   al   su   age    bp    bgr     bu    sc    sod  pot  ...  \
0       102677  3.0  0.0  59.0  70.0   76.0  186.0  15.0  135.0  7.6  ...   
1       337834  4.0  0.0  48.0  70.0  117.0   56.0   3.8  111.0  2.5  ...   
2       343710  2.0  0.0  61.0  80.0  173.0  148.0   3.9  135.0  5.2  ...   
3       349892  1.0  3.0  59.0  70.0  424.0   55.0   1.7  138.0  4.5  ...   
4       353025  3.0  4.0  40.0  70.0  253.0  150.0  11.9  132.0  5.6  ...   
..         ...  ...  ...   ...   ...    ...    ...   ...    ...  ...  ...   
148     486397  0.0  0.0  20.0  70.0  123.0   44.0   1.0  135.0  3.8  ...   
149     203466  0.0  0.0  22.0  60.0   97.0   18.0   1.2  138.0  4.3  ...   
150     482105  0.0  0.0  44.0  60.0   95.0   46.0   0.5  138.0  4.2  ...   
151     228849  0.0  0.0  58.0  80.0  100.0   50.0   1.2  140.0  3.5  ...   
152     996076  0.0  0.0  57.0  60.0  105.0   49.0   1.2  150.0  4.7  ...   

     cad_no  cad_yes  appet_good  appet_poor  pe_no  pe_yes  ane_no  ane_ye

In the example we went through above, another solution is to have a single column for the binary variable. In the downstream modeling would this be equivalent? What effect would this have if the categorical variable could take more than 2 values? For example, let's say we have a categorical feature that is "type of condiment" that can take 5 separate values and we are trying to predict the rating of a particular sandwich.

Answer: We would lose information. This is because if there were more than two possible values, such as in the sandwich example, we can only record two of the possible five values for that column, so if the data is one of the three other values, it is lost.

## Part 3.6 : Remove Outliers from Numerical Columns

Outliers can disproportionately influence the fit of a regression model, potentially leading to a model that does not generalize well therefore it is important that we remove outliers from the numerical columns of the dataset.

For this dataset, we define an outlier to be 3 times the standard deviation from the mean. Drop these outliers from the dataset

In [228]:
numerical_columns = ['age', 'bp', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc']

# Remove outliers
for column in numerical_columns:
    mean = cleaned_df[column].mean()
    std_dev = cleaned_df[column].std()
    cleaned_df = cleaned_df[(cleaned_df[column] >= mean - 3 * std_dev) & (cleaned_df[column] <= mean + 3 * std_dev)]

print(cleaned_df)


     unique_id   al   su   age     bp    bgr     bu   sc    sod  pot  ...  \
2       343710  2.0  0.0  61.0   80.0  173.0  148.0  3.9  135.0  5.2  ...   
5       397388  2.0  2.0  63.0  100.0  280.0   35.0  3.2  143.0  3.5  ...   
7       438182  2.0  0.0  73.0   80.0  253.0  142.0  4.6  138.0  5.8  ...   
8       474407  4.0  0.0  21.0   90.0  107.0   40.0  1.7  125.0  3.5  ...   
9       475200  4.0  2.0  64.0  100.0  163.0   54.0  7.2  140.0  4.6  ...   
..         ...  ...  ...   ...    ...    ...    ...  ...    ...  ...  ...   
148     486397  0.0  0.0  20.0   70.0  123.0   44.0  1.0  135.0  3.8  ...   
149     203466  0.0  0.0  22.0   60.0   97.0   18.0  1.2  138.0  4.3  ...   
150     482105  0.0  0.0  44.0   60.0   95.0   46.0  0.5  138.0  4.2  ...   
151     228849  0.0  0.0  58.0   80.0  100.0   50.0  1.2  140.0  3.5  ...   
152     996076  0.0  0.0  57.0   60.0  105.0   49.0  1.2  150.0  4.7  ...   

     cad_no  cad_yes  appet_good  appet_poor  pe_no  pe_yes  ane_no  ane_ye

## Part 3.7 : Normalize the Numerical Columns

Normalizing numerical attributes ensures that all features contribute equally to the model by scaling them to a consistent range, which improves model performance and convergence. It prevents features with larger scales from disproportionately influencing the model's learning process.

In [229]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
cleaned_df[numerical_columns] = scaler.fit_transform(cleaned_df[numerical_columns])

print(cleaned_df)


     unique_id   al   su       age    bp       bgr        bu        sc  \
2       343710  2.0  0.0  0.743243  0.50  0.442060  0.901961  0.432099   
5       397388  2.0  2.0  0.770270  1.00  0.901288  0.163399  0.345679   
7       438182  2.0  0.0  0.905405  0.50  0.785408  0.862745  0.518519   
8       474407  4.0  0.0  0.202703  0.75  0.158798  0.196078  0.160494   
9       475200  4.0  2.0  0.783784  1.00  0.399142  0.287582  0.839506   
..         ...  ...  ...       ...   ...       ...       ...       ...   
148     486397  0.0  0.0  0.189189  0.25  0.227468  0.222222  0.074074   
149     203466  0.0  0.0  0.216216  0.00  0.115880  0.052288  0.098765   
150     482105  0.0  0.0  0.513514  0.00  0.107296  0.235294  0.012346   
151     228849  0.0  0.0  0.702703  0.50  0.128755  0.261438  0.098765   
152     996076  0.0  0.0  0.689189  0.00  0.150215  0.254902  0.098765   

          sod       pot  ...  cad_no  cad_yes  appet_good  appet_poor  pe_no  \
2    0.500000  0.793103  ...   

## Part 3.8: Remove Unnecessary columns

Are there any columns in this dataset which are not appropriate for modeling and predictions? Which column(s)? Justify their exclusion and remove them

Answer: The "unique_id" column is unnessicary becasue it only contains an identifying number for each patient. The ID number is only to identify each patient, it is not actual data from the patient, and it is likely randomly generated, or generated based on something irrelevent. Because if this, the ID number should be removed because it will not be useful for whatever we are trying to predict with this data set.

In [230]:
# Remove that column
cleaned_df = cleaned_df.drop(columns=['unique_id'])

# Print the dataset
print(cleaned_df)


      al   su       age    bp       bgr        bu        sc       sod  \
2    2.0  0.0  0.743243  0.50  0.442060  0.901961  0.432099  0.500000   
5    2.0  2.0  0.770270  1.00  0.901288  0.163399  0.345679  0.766667   
7    2.0  0.0  0.905405  0.50  0.785408  0.862745  0.518519  0.600000   
8    4.0  0.0  0.202703  0.75  0.158798  0.196078  0.160494  0.166667   
9    4.0  2.0  0.783784  1.00  0.399142  0.287582  0.839506  0.666667   
..   ...  ...       ...   ...       ...       ...       ...       ...   
148  0.0  0.0  0.189189  0.25  0.227468  0.222222  0.074074  0.500000   
149  0.0  0.0  0.216216  0.00  0.115880  0.052288  0.098765  0.600000   
150  0.0  0.0  0.513514  0.00  0.107296  0.235294  0.012346  0.600000   
151  0.0  0.0  0.702703  0.50  0.128755  0.261438  0.098765  0.666667   
152  0.0  0.0  0.689189  0.00  0.150215  0.254902  0.098765  1.000000   

          pot      hemo  ...  cad_no  cad_yes  appet_good  appet_poor  pe_no  \
2    0.793103  0.000000  ...   False     Tr

## Part 3.9: Export the Cleaned Data

Now that you've completed these cleaning steps you should have a pandas dataframe which is much cleaner and ready for modeling. Our final step is to save our work. Export the DataFrame to a two new formats: csv and json.

In [231]:
# Export the dataframe to a new csv file
cleaned_df.to_csv('cleaned_data.csv', index=False)

# Export the dataframe to a new json file
cleaned_df.to_json('cleaned_data.json', orient='records')

# Part 4: Data conversions with Large Language Models

One powerful use case of ChatGPT (and other generative language models) is cleaning and transforming data. In some cases, these models can directly manipulate loosely structured data that you provide to them into a standard format. In the other cases, you can often prompt the model to create a conversion or extraction script for you in python or Pandas and then run it on your own. 

In this part of the assignment you will prompt 383GPT to explore these capabilities.

## Part 4.1 GPT Data Manipulation

Take the cleaned dataset that you created in part three and output the top 15 rows of that dataset. Then copy the terminal output, open 383gpt and ask it to convert that output to a markdown table. Paste that markdown table in the cell bellow

Paste here: 

| al   | su   | age                | bp   | bgr                 | bu                  | sc                 | sod                | pot                | hemo                | pcv                 | wbcc                | rbcc                | rbc_normal | pc_normal | pcc_present | ba_present | htn_yes | dm_yes | cad_yes | appet_poor | pe_yes | ane_yes | Target_notckd |
|------|------|--------------------|------|---------------------|---------------------|--------------------|--------------------|--------------------|---------------------|---------------------|---------------------|---------------------|------------|-----------|-------------|------------|---------|--------|---------|------------|--------|---------|-----------------|
| 2.0  | 0.0  | 0.7432432432432432 | 0.5  | 0.44206008583690987 | 0.9019607843137255  | 0.4320987654320988  | 0.5                | 0.7931034482758623 | 0.0                 | 0.032258064516129004 | 0.3951612903225806 | 0.05714285714285716 | False      | False     | False       | False      | True    | True   | True    | True       | True   | True    | False           |
| 2.0  | 2.0  | 0.7702702702702704 | 1.0  | 0.9012875536480687 | 0.16339869281045755 | 0.34567901234567905 | 0.7666666666666666 | 0.2068965517241379 | 0.5247524752475247  | 0.5483870967741935 | 0.44354838709677413 | 0.34285714285714286 | True       | True      | False       | True       | True    | False  | True    | False      | False   | False   | False           |
| 2.0  | 0.0  | 0.9054054054054055 | 0.5  | 0.7854077253218883 | 0.8627450980392157  | 0.5185185185185185  | 0.5999999999999996 | 1.0                | 0.27722772277227714 | 0.32258064516129026 | 0.2338709677419355 | 0.37142857142857133 | False      | False     | False       | False      | True    | True   | True    | False      | False   | False   | False           |
| 4.0  | 0.0  | 0.2027027027027027 | 0.75 | 0.15879828326180256 | 0.19607843137254904 | 0.16049382716049385 | 0.16666666666666696 | 0.2068965517241379 | 0.05940594059405946 | 0.0                 | 0.6532258064516129 | 0.257142857142857  | True       | False     | True        | True       | False   | False  | False   | False      | False   | True    | False           |
| 4.0  | 2.0  | 0.7837837837837838 | 1.0  | 0.39914163090128757 | 0.28758169934640526 | 0.8395061728395062  | 0.666666666666667  | 0.5862068965517242 | 0.01980198019801982 | 0.09677419354838701 | 0.25806451612903225 | 0.11428571428571421 | False      | False     | False       | True       | True    | True   | False   | False      | True    | False   | False           |
| 3.0  | 1.0  | 0.7297297297297298 | 0.0  | 0.9356223175965666 | 0.16993464052287582 | 0.16049382716049385 | 0.33333333333333304 | 0.034482758620689724 | 0.01980198019801982 | 0.06451612903225801 | 0.8790322580645161 | 0.0                 | True       | False     | True        | False      | True    | False  | False   | True       | False   | True    | False           |
| 2.0  | 0.0  | 0.6756756756756757 | 0.75 | 0.25321888412017163 | 0.6339869281045752  | 0.7777777777777779  | 0.36666666666666625 | 0.6551724137931034 | 0.13861386138613863 | 0.19354838709677413 | 0.16935483870967738 | 0.11428571428571421 | False      | False     | False       | False      | True    | False  | False   | False      | False   | False   | False           |
| 4.0  | 3.0  | 0.8513513513513513 | 0.25 | 0.6180257510729614 | 0.5620915032679739  | 0.7283950617283951  | 0.0                | 0.3448275862068966 | 0.16831683168316836 | 0.16129032258064513 | 0.5806451612903225 | 0.08571428571428563 | True       | False     | True        | True       | True    | True   | True    | False      | True    | True    | False           |
| 1.0  | 0.0  | 0.5405405405405406 | 0.0  | 0.39914163090128757 | 0.5359477124183006  | 0.35802469135802467 | 0.7000000000000002  | 0.3793103448275863 | 0.207920792079208  | 0.16129032258064513 | 0.8306451612903226 | 0.05714285714285716 | True       | True      | False       | False      | True    | True   | False   | False      | False   | False   | False           |
| 3.0  | 0.0  | 0.7567567567567568 | 0.25 | 0.22317596566523606 | 0.20915032679738566 | 0.16049382716049385 | 0.5333333333333332  | 0.6206896551724139 | 0.485148514851485  | 0.5161290322580645 | 0.29032258064516125 | 0.257142857142857  | True       | False     | False       | False      | True    | True   | False   | False      | False   | False   | False           |
| 3.0  | 1.0  | 0.6621621621621623 | 0.5  | 0.6180257510729614 | 0.411764705882353  | 0.4320987654320988  | 0.5666666666666664  | 0.6896551724137934 | 0.3168316831683168 | 0.35483870967741926 | 0.25                | 0.20000000000000007 | True       | False     | True        | True       | True    | True   | False   | False      | True    | False   | False           |
| 4.0  | 1.0  | 0.7837837837837838 | 0.0  | 0.7253218884120172 | 0.3137254901960784  | 0.4814814814814815  | 0.5666666666666664  | 0.8620689655172415 | 0.17821782178217827 | 0.19354838709677413 | 0.25806451612903225 | 0.11428571428571421 | False      | False     | False       | True       | True    | True   | False   | True       | True    | False   | False           |
| 4.0  | 0.0  | 0.8783783783783785 | 0.0  | 0.20600858369098712 | 0.7516339869281046  | 0.6049382716049383  | 0.5333333333333332  | 0.6896551724137934 | 0.36633663366336633 | 0.38709677419354827 | 0.8790322580645161 | 0.37142857142857133 | True       | True      | False       | False      | True    | True   | False   | True       | True    | False   | False           |
| 3.0  | 0.0  | 0.8783783783783785 | 0.25 | 0.6394849785407726  | 0.47058823529411764 | 0.39506172839506176 | 0.43333333333333357 | 0.517241379310345  | 0.26732673267326745 | 0.32258064516129026 | 0.10483870967741932 | 0.17142857142857137 | True       | False     | True        | True       | True    | True   | True    | False      | False   | False   | False           |


** Caution: ** while language models can perform data conversions they also can * hallucinate * during this process, particularly for bigger datasets. Reflect on this below, how could you mitigate data conversion hallucinations from LLM conversions?

When prompting the llm you could mitigate hallucination by ensuring that the prompt is as clear as possible. You could also give context about the prompt so that the llm doesn't try to use any extraneous data in the conversion.

## Part 4.2 GPT Pandas Prompting

In this section, you will prompt 383GPT to write pandas code manipulations for you.

After working with this data for awhile, we realized we're starting to forget the meanings of the abbreviated column names. Let's ask 383GPT to fix this for us. First, navigate to the [UCI dataset overview](https://archive.ics.uci.edu/dataset/336/chronic+kidney+disease) and copy the abbrevation to name mapping. Then, go to 383GPT and instruct the LLM to provide you with a pandas script to apply this renaming to all the columns of your dataset. Paste that code below and make any adjustments necessary to run it in your notebook.

In [232]:
# Code to rename all the columns in the dataset
import pandas as pd

# Assuming your DataFrame is named 'df'
# Create a mapping dictionary
column_mapping = {
    'age': 'age',
    'bp': 'blood pressure',
    'sg': 'specific gravity',
    'al': 'albumin',
    'su': 'sugar',
    'rbc': 'red blood cells',
    'pc': 'pus cell',
    'pcc': 'pus cell clumps',
    'ba': 'bacteria',
    'bgr': 'blood glucose random',
    'bu': 'blood urea',
    'sc': 'serum creatinine',
    'sod': 'sodium',
    'pot': 'potassium',
    'hemo': 'hemoglobin',
    'pcv': 'packed cell volume',
    'wc': 'white blood cell count',
    'rc': 'red blood cell count',
    'htn': 'hypertension',
    'dm': 'diabetes mellitus',
    'cad': 'coronary artery disease',
    'appet': 'appetite',
    'pe': 'pedal edema',
    'ane': 'anemia',
    'class': 'class'
}

# Rename the columns using the mapping
cleaned_df.rename(columns=column_mapping, inplace=True)

# Display the updated DataFrame (optional)
print(cleaned_df)

     albumin  sugar       age  blood pressure  blood glucose random  \
2        2.0    0.0  0.743243            0.50              0.442060   
5        2.0    2.0  0.770270            1.00              0.901288   
7        2.0    0.0  0.905405            0.50              0.785408   
8        4.0    0.0  0.202703            0.75              0.158798   
9        4.0    2.0  0.783784            1.00              0.399142   
..       ...    ...       ...             ...                   ...   
148      0.0    0.0  0.189189            0.25              0.227468   
149      0.0    0.0  0.216216            0.00              0.115880   
150      0.0    0.0  0.513514            0.00              0.107296   
151      0.0    0.0  0.702703            0.50              0.128755   
152      0.0    0.0  0.689189            0.00              0.150215   

     blood urea  serum creatinine    sodium  potassium  hemoglobin  ...  \
2      0.901961          0.432099  0.500000   0.793103    0.000000  ... 

## Part 4.3 Augmenting our skills with prompting

In addition, we can also use 383GPT to convert our data manipulation operations between different data manipulation languages and libraries. For example let's prompt 383GPT to convert the following SQL query to a pandas query.

**SQL Query**
```sql
SELECT Target, COUNT(*) AS count
FROM your_table_name
GROUP BY Target;
```

Prompt 383GPT to convert this to a pandas query. Run this query below, then describe what it does. (If you're not familiar with SQL that is okay you need to only comment on the final resulting output.)

In [233]:
# Converted SQL to Pandas code
import pandas as pd

# Replace 'your_table_name' with your actual DataFrame name
# df = pd.read_csv('your_data_file.csv')  # Example of loading data

# Group by 'Target' and count the occurrences
result = cleaned_df.groupby('Target_ckd').size().reset_index(name='count')

# Display the result
print(result)

   Target_ckd  count
0       False    111
1        True     22


*Describe what the above code does here*

It found how many patients in the cleaned data set have ckd, and how many don't.